## Load Libraries

In [ ]:
import json
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
from transformers import Trainer
from transformers import TrainingArguments
from transformers import AutoTokenizer
from collections import Counter
from datasets import Dataset

# Retierver and paper selection

## load DataSet

In [ ]:
def load_corpus(path):
    papers = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            paper = json.loads(line)
            papers.append(paper)
    return papers

corpus = load_corpus("data/corpus.jsonl")
print(len(corpus))
corpus[0]

5183


{'doc_id': 4983,
 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.',
 'abstract': ['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.',
  'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).',
  'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.',
  'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.',
  'In the posterior limb 

In [ ]:
print(corpus[0]["doc_id"])
print(corpus[0]["title"])
print(corpus[0]["abstract"])

4983
Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
['Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.', 'A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).', 'To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.', 'In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms.', 'In the posterior limb of the internal capsule, the mean apparent dif

## Data Inspection

In [ ]:
print("Corpus abstract type : ", type(corpus[0]["abstract"]))
#sentence number
print("sentence number : " , len(corpus[0]["abstract"]))
print("First sentence : ", corpus[0]["abstract"][0])

Corpus abstract type :  <class 'list'>
sentence number :  14
First sentence :  Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.


In [ ]:
def prepare_document(paper):
    title = paper["title"]
    abstract = " ".join(paper["abstract"])
    document_text = title + " " + abstract
    return document_text

document = prepare_document(corpus[0])
document

'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficient

In [ ]:
#make the whole corpus
documents = []
for paper in corpus:
    document = prepare_document(paper)
    documents.append(document)

print(len(documents))

5183


##  TF-IDF Retrieval

In [ ]:
class TFIDFRetriever:

    def __init__(self, corpus):
        self.corpus = corpus
        self.documents = documents
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2)
        )
        self.document_vectors = self.vectorizer.fit_transform(
            self.documents
        )
    def retrieve(self, claim, top_k=5):
        claim_vector = self.vectorizer.transform([claim])
        scores = cosine_similarity(
            claim_vector,
            self.document_vectors
        )[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for index in top_indices:
            paper = self.corpus[index]
            results.append({
                "doc_id": paper["doc_id"],
                "title": paper["title"],
                "score": float(scores[index]),
                "abstract": paper["abstract"]
            })
        return results

In [ ]:
#test Tf-idf
corpus = load_corpus("data/corpus.jsonl")
tfidf_retriever = TFIDFRetriever(corpus)
claim = "Prematurity affects cerebral white matter development."
results = tfidf_retriever.retrieve(
        claim,
        top_k=5
    )
print("\n===== TF-IDF RESULTS =====")
for result in results:
    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])


===== TF-IDF RESULTS =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 0.408817003859458

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 0.2506619103513072

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 0.187248318336902

Doc ID: 17930286
Title: Headache, migraine, and structural brain lesions and function: population based Epidemiology of Vascular Ageing-MRI study
Score: 0.14636511997231055

Doc ID: 1472815
Title: Alterations of white matter integrity in adults with major depressive disorder: a magnetic resonance imaging study.
Score: 0.14410076468240382


## BM25Retriever

In [ ]:
def tokenize(text):
    text = text.lower()
    tokens = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )
    return tokens

In [ ]:
class BM25Retriever:
    def __init__(self, corpus):
        self.corpus = corpus
        self.documents = documents
        self.tokenized_documents = [
            tokenize(document)
            for document in self.documents
        ]
        self.bm25 = BM25Okapi(
            self.tokenized_documents
        )
    def retrieve(self, claim, top_k=5):
        query_tokens = tokenize(claim)
        scores = self.bm25.get_scores(
            query_tokens
        )
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for index in top_indices:
            paper = self.corpus[index]
            results.append({
                "doc_id": paper["doc_id"],
                "title": paper["title"],
                "score": float(scores[index]),
                "abstract": paper["abstract"]
            })

        return results

In [ ]:
#test BM25
corpus = load_corpus("data/corpus.jsonl")
bm25_retriever = BM25Retriever(corpus)
claim = "Prematurity affects cerebral white matter development."
results = bm25_retriever.retrieve(
        claim,
        top_k=5
    )

print("\n===== BM25 RESULTS =====")
for result in results:
    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])


===== BM25 RESULTS =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 36.59155958791644

Doc ID: 8227227
Title: Locations of cerebral infarctions in tuberculous meningitis
Score: 20.258535179588897

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 18.356174616468792

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 17.695535039671007

Doc ID: 18104691
Title: Neurological outcomes of animal models of uterine artery ligation and relevance to human intrauterine growth restriction: a systematic review
Score: 17.67830572816744


## Semantic retrieval

In [ ]:
class SemanticRetriever:
    def __init__(self, corpus):
        self.corpus = corpus
        self.documents = documents
        print("Loading semantic model...")
        self.model = SentenceTransformer(
            "allenai-specter"
        )
        print("Creating document embeddings...")
        self.document_embeddings = self.model.encode(
            self.documents,
            convert_to_numpy=True,
            show_progress_bar=True
        )

    def retrieve(self, claim, top_k=5):
        claim_embedding = self.model.encode(
            [claim],
            convert_to_numpy=True
        )
        scores = cosine_similarity(
            claim_embedding,
            self.document_embeddings)[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for index in top_indices:
            paper = self.corpus[index]
            results.append({
                "doc_id": paper["doc_id"],
                "title": paper["title"],
                "score": float(scores[index]),
                "abstract": paper["abstract"]
            })
        return results

## Evaluate the Retrieval algorithms

In [ ]:
claim = "Prematurity affects cerebral white matter development."
# TF-IDF
print("\n\n===== TF-IDF =====")
tfidf_retriever = TFIDFRetriever(corpus)
tfidf_results = tfidf_retriever.retrieve(
    claim,
    top_k=5
)
for result in tfidf_results:
    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])





===== TF-IDF =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 0.408817003859458

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 0.2506619103513072

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 0.187248318336902

Doc ID: 17930286
Title: Headache, migraine, and structural brain lesions and function: population based Epidemiology of Vascular Ageing-MRI study
Score: 0.14636511997231055

Doc ID: 1472815
Title: Alterations of white matter integrity in adults with major depressive disorder: a magnetic resonance imaging study.
Score: 0.14410076468240382


In [ ]:
# BM25
print("\n\n===== BM25 =====")
bm25_retriever = BM25Retriever(corpus)
bm25_results = bm25_retriever.retrieve(
    claim,
    top_k=5
)
for result in bm25_results:
    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Score:", result["score"])




===== BM25 =====

Doc ID: 4983
Title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
Score: 36.59155958791644

Doc ID: 8227227
Title: Locations of cerebral infarctions in tuberculous meningitis
Score: 20.258535179588897

Doc ID: 1412089
Title: Diminished performance on neuropsychological testing in late life depression is correlated with microstructural white matter abnormalities.
Score: 18.356174616468792

Doc ID: 22107641
Title: Late-life depression and microstructural abnormalities in dorsolateral prefrontal cortex white matter.
Score: 17.695535039671007

Doc ID: 18104691
Title: Neurological outcomes of animal models of uterine artery ligation and relevance to human intrauterine growth restriction: a systematic review
Score: 17.67830572816744


## Evalution metrics

In [ ]:
def get_gold_documents(claim):
    evidence = claim.get("evidence", {})
    gold_docs = set()
    for doc_id in evidence.keys():
        gold_docs.add(int(doc_id))
    return gold_docs

def recall_at_k(retrieved_results, gold_docs, k):
    retrieved_docs = {
        result["doc_id"]
        for result in retrieved_results[:k]
    }
    if len(gold_docs.intersection(retrieved_docs)) > 0:
        return 1
    return 0

def reciprocal_rank(retrieved_results, gold_docs):
    for rank, result in enumerate(
        retrieved_results,
        start=1
    ):
        if result["doc_id"] in gold_docs:
            return 1 / rank
    return 0

In [ ]:
def evaluate_retriever(retriever,claims,max_k=5):
    recall_1 = []
    recall_5 = []
    reciprocal_ranks = []
    for claim in claims:
        gold_docs = get_gold_documents(claim)
        if not gold_docs:
            continue
        results = retriever.retrieve(
            claim["claim"],
            top_k=max_k
        )
        recall_1.append(
            recall_at_k(
                results,
                gold_docs,
                1
            )
        )
        recall_5.append(
            recall_at_k(
                results,
                gold_docs,
                5))

        reciprocal_ranks.append(
            reciprocal_rank(
                results,
                gold_docs
            )
        )

    return {
        "Recall@1": np.mean(recall_1),
        "Recall@5": np.mean(recall_5),
        "MRR": np.mean(reciprocal_ranks)
    }

In [ ]:
def load_claims(path):
    with open(path, "r", encoding="utf-8") as file:
        content = file.read().strip()
    claims = []
    for line in content.splitlines():
        if line.strip():
            claims.append(
                json.loads(line)
            )
    return claims

claims = load_claims("data/claims_dev.jsonl")

In [ ]:
# print("\n===== EVALUATION =====")
# tfidf_metrics = evaluate_retriever(
#     tfidf_retriever,
#     claims
# )
# bm25_metrics = evaluate_retriever(
#     bm25_retriever,
#     claims
# )
# semantic_metrics = evaluate_retriever(
#     semantic_retriever,
#     claims
# )

# print("\nTF-IDF:")
# print(tfidf_metrics)

# print("\nBM25:")
# print(bm25_metrics)

# print("\nSemantic:")
# print(semantic_metrics)

# Evidence selection

In [ ]:
claims_dev = load_claims("data/claims_dev.jsonl")
claims_trains = load_claims("data/claims_train.jsonl")

In [ ]:
corpus_by_id ={
    paper["doc_id"] : paper for paper in corpus
}
print(len(corpus_by_id))

5183


In [ ]:
def score_sentences_with_bm25(claim, paper):
    sentences = paper["abstract"]
    tokenized_sentences = [
        tokenize(sentence)
        for sentence in sentences
    ]
    bm25 = BM25Okapi(tokenized_sentences)
    query_tokens = tokenize(claim)
    scores = bm25.get_scores(query_tokens)
    results = []
    for idx, sentence in enumerate(sentences):
        results.append({
            "sentence_idx": idx,
            "sentence": sentence,
            "bm25_score": float(scores[idx])
        })
    return results

In [ ]:
# #function to add the document BM25 scores
# def get_document_bm25_scores(claims, bm25_retriever):
#     claim_doc_scores = {}
#     for claim in claims:
#         claim_text = claim["claim"]
#         query_tokens = tokenize(claim_text)
#         scores = bm25_retriever.bm25.get_scores(query_tokens)
#         claim_doc_scores[claim_text] = {
#             corpus[i]["doc_id"]: float(scores[i])
#             for i in range(len(corpus))
#         }

#     return claim_doc_scores

In [ ]:
# train_doc_scores = get_document_bm25_scores(
#     claims_trains,
#     bm25_retriever
# )

# dev_doc_scores = get_document_bm25_scores(
#     claims_dev,
#     bm25_retriever
# )

In [ ]:
def build_evidence_pairs(claims, corpus_by_id):
    rows = []
    for claim in claims:
        evidence = claim.get("evidence", {})
        if not evidence:
            continue
        for doc_id_str, evidence_list in evidence.items():
            doc_id = int(doc_id_str)
            paper = corpus_by_id.get(doc_id)
            if paper is None:
                continue
            gold_sentences = set()
            for evidence_item in evidence_list:
                gold_sentences.update(
                    evidence_item["sentences"]
                )
            for idx, sentence in enumerate(paper["abstract"]):
                rows.append({
                    "claim": claim["claim"],
                    "sentence": sentence,
                    "doc_id": doc_id,
                    "sentence_idx": idx,
                    #"doc_bm25_scores":doc_scores[claim["claim"]][doc_id],
                    "label": 1 if idx in gold_sentences else 0
                })

    return rows

In [ ]:
train_rows = build_evidence_pairs(claims_trains,corpus_by_id)
dev_rows = build_evidence_pairs(claims_dev,corpus_by_id )

print("Train pairs:", len(train_rows))
print("Train positives:", sum(r["label"] for r in train_rows))

print("Dev pairs:", len(dev_rows))
print("Dev positives:", sum(r["label"] for r in dev_rows))

Train pairs: 5494
Train positives: 1025
Dev pairs: 2031
Dev positives: 366


In [ ]:
#word overlap
def word_overlap(a, b):
    words_a = set(tokenize(a))
    words_b = set(tokenize(b))
    if not words_a or not words_b:
        return 0.0

    return len(words_a & words_b) / len(words_a | words_b)

In [ ]:
#calculate bm25 for each sentence in the sentences
def add_bm25_sentence_scores(rows, corpus_by_id):
    scores_cache = {}
    for doc_id in set(row["doc_id"] for row in rows):
        paper = corpus_by_id[doc_id]
        sentences = paper["abstract"]
        tokenized_sentences = [
            tokenize(sentence)
            for sentence in sentences
        ]
        bm25 = BM25Okapi(tokenized_sentences)
        scores_cache[doc_id] = {}
        for row in rows:
            if row["doc_id"] != doc_id:
                continue
            query_tokens = tokenize(row["claim"])
            scores = bm25.get_scores(query_tokens)
            row["bm25_score"] = float(
                scores[row["sentence_idx"]]
            )

    return rows

In [ ]:
train_rows = add_bm25_sentence_scores(
    train_rows,
    corpus_by_id
)

dev_rows = add_bm25_sentence_scores(
    dev_rows,
    corpus_by_id
)

In [ ]:
def featurize_evidence_rows(rows):
    X = []
    y = []
    for row in rows:
        overlap = word_overlap(
            row["claim"],
            row["sentence"]
        )

        position = row["sentence_idx"]
        length = len(
            row["sentence"].split()
        )

        X.append([
           # row["doc_bm25_scores"],
            row["bm25_score"],
            overlap,
            position,
            length
        ])

        y.append(row["label"])

    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = featurize_evidence_rows(
    train_rows
)

X_dev, y_dev = featurize_evidence_rows(
    dev_rows
)

print(X_train.shape)
print(X_dev.shape)

(5494, 4)
(2031, 4)


In [ ]:
evidence_model = Pipeline([
    #use the scaler to avoid bias
    #("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

evidence_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [ ]:
#random forest
from sklearn.ensemble import RandomForestClassifier

evidence_random_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

evidence_random_model.fit(
    X_train,
    y_train
)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
import joblib
joblib.dump(
    evidence_random_model,
    "evidence_selector_rf.joblib"
)

In [ ]:
evidence_random_model= joblib.load(
    "evidence_selector_rf.joblib"
)

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


#### Evalute model before adding document BM25

In [ ]:
y_pred = evidence_model.predict(X_dev)
print(classification_report(
        y_dev,
        y_pred,
        target_names=[
            "Not Evidence",
            "Evidence"
        ]
    )
)

              precision    recall  f1-score   support

Not Evidence       0.89      0.70      0.78      1665
    Evidence       0.31      0.62      0.42       366

    accuracy                           0.69      2031
   macro avg       0.60      0.66      0.60      2031
weighted avg       0.79      0.69      0.72      2031



#### after adding document bm25

In [ ]:
y_pred = evidence_random_model.predict(X_dev)
print(classification_report(
        y_dev,
        y_pred,
        target_names=[
            "Not Evidence",
            "Evidence"
        ]
    )
)

              precision    recall  f1-score   support

Not Evidence       0.88      0.97      0.93      1665
    Evidence       0.76      0.42      0.54       366

    accuracy                           0.87      2031
   macro avg       0.82      0.69      0.73      2031
weighted avg       0.86      0.87      0.86      2031



In [ ]:
def select_evidence(claim,retrieved_papers,model, top_n=3):
    candidates = []
    for paper in retrieved_papers:
        sentences = paper["abstract"]
        tokenized_sentences = [
            tokenize(sentence)
            for sentence in sentences
        ]
        bm25 = BM25Okapi(
            tokenized_sentences
        )

        query_tokens = tokenize(claim)
        bm25_scores = bm25.get_scores(
            query_tokens
        )

        for idx, sentence in enumerate(sentences):
            overlap = word_overlap(
                claim,
                sentence
            )
            position = idx
            length = len(
                sentence.split()
            )
            features = np.array([[
               # paper["score"],
                bm25_scores[idx],
                overlap,
                position,
                length
            ]])

            probability = model.predict_proba(
                features
            )[0][1]

            candidates.append({
               "doc_id": paper["doc_id"],
                "title": paper["title"],
                "sentence_idx": idx,
                "sentence": sentence,
                "bm25_score": float(
                    bm25_scores[idx]
                ),
                "probability": float(
                    probability
                )
            })

    candidates.sort(
        key=lambda x: x["probability"],
        reverse=True
    )

    return candidates[:top_n]

In [ ]:
example_claim = load_claims("data/claims_dev.jsonl")[0]["claim"]
print(example_claim)

0-dimensional biomaterials show inductive properties.


#### BM25 for sentence selection

In [ ]:
bm25_results = bm25_retriever.retrieve(
    example_claim,
    top_k=5
)
evidence_results = select_evidence(
    example_claim,
    bm25_results,
    evidence_model,
    top_n=3
)
for result in evidence_results:

    print("\nDoc ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Sentence:", result["sentence"])
    print("BM25:", result["bm25_score"])
    print("Evidence probability:", result["probability"])


Doc ID: 40212412
Title: Periosteal bone formation--a neglected determinant of bone strength.
Sentence: This need is met by the lever function of long bones, three-dimensional masterpieces of biomechanical engineering that, by their material composition and structural design, achieve the contradictory properties of stiffness and flexibility, strength and lightness.1 Material stiffness results from the encrusting of the triple-helical structure of collagen type I with hydroxyapatite crystals, which confers . . .
BM25: 0.7956905746373846
Evidence probability: 0.6861302380487918

Doc ID: 10906636
Title: The carboxyl terminus of human cytomegalovirus-encoded 7 transmembrane receptor US28 camouflages agonism by mediating constitutive endocytosis.
Sentence: We further show that the constitutive endocytic property of US28 affects the action of its chemokine ligand fractalkine/CX3CL1 and show that in the absence of the US28 C terminus, fractalkine/CX3CL1 acts as an agonist on US28.
BM25: 0.598

## Evidence Recall@1 / @3 / @5

In [ ]:
def get_gold_evidence_senteces(claim):
    gold_sentences = set()
    evidence = claim.get("evidence", {})
    for doc_id , evidence_list in evidence.items():
        doc_id = int(doc_id)
        for evidence_item in evidence_list :
            for sentence_idx in evidence_item["sentences"]:
                gold_sentences.add((doc_id , sentence_idx))
    return gold_sentences

In [ ]:
def evidence_recall_at_k(results, gold_evidence, k):
    retrieved_evidence = {
        (result["doc_id"], result["sentence_idx"])
        for result in results[:k]
    }
    if gold_evidence.intersection(retrieved_evidence):
        return 1
    return 0

In [ ]:
def evaluate_evidence_selector(
    claims,
    retriever,
    evidence_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=3
):

    recall_scores = []

    for claim in claims:

        gold_evidence = get_gold_evidence_senteces(
            claim
        )

        if not gold_evidence:
            continue

        retrieved_papers = retriever.retrieve(
            claim["claim"],
            top_k=retrieval_k
        )

        selected_evidence = select_evidence(
            claim["claim"],
            retrieved_papers,
            evidence_model,
            top_n=evidence_k
        )

        score = evidence_recall_at_k(
            selected_evidence,
            gold_evidence,
            evidence_k
        )

        recall_scores.append(score)

    return np.mean(recall_scores)

### Evalution without using the document bm25

In [ ]:
recall_1 = evaluate_evidence_selector(
    claims_dev,
    bm25_retriever,
    evidence_random_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=1
)

recall_3 = evaluate_evidence_selector(
    claims_dev,
    bm25_retriever,
    evidence_random_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=3
)

recall_5 = evaluate_evidence_selector(
    claims_dev,
    bm25_retriever,
    evidence_random_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=5
)
print("Evidence Recall@1:", recall_1)
print("Evidence Recall@3:", recall_3)
print("Evidence Recall@5:", recall_5)

Evidence Recall@1: 0.35106382978723405
Evidence Recall@3: 0.4627659574468085
Evidence Recall@5: 0.5319148936170213


In [ ]:
recall_1 = evaluate_evidence_selector(
    claims_dev,
    bm25_retriever,
    evidence_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=1
)

recall_3 = evaluate_evidence_selector(
    claims_dev,
    bm25_retriever,
    evidence_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=3
)

recall_5 = evaluate_evidence_selector(
    claims_dev,
    bm25_retriever,
    evidence_model,
    corpus_by_id,
    retrieval_k=5,
    evidence_k=5
)

print("Evidence Recall@1:", recall_1)
print("Evidence Recall@3:", recall_3)
print("Evidence Recall@5:", recall_5)

Evidence Recall@1: 0.25
Evidence Recall@3: 0.4095744680851064
Evidence Recall@5: 0.526595744680851


# Fine tuning & model Selection

In [ ]:
def build_nli_examples(
    claims,
    corpus_by_id,
    max_negative_per_claim=3
):

    examples = []

    for claim in claims:

        claim_text = claim["claim"]

        evidence = claim.get(
            "evidence",
            {}
        )
        # Positive / Gold Evidence

        gold_sentence_pairs = set()

        for doc_id_str, ev_list in evidence.items():

            doc_id = int(doc_id_str)

            paper = corpus_by_id.get(doc_id)

            if paper is None:
                continue

            for ev in ev_list:

                label = ev["label"]

                sent_ids = ev["sentences"]

                evidence_text = " ".join(
                    paper["abstract"][i]
                    for i in sent_ids
                )

                examples.append({
                    "claim": claim_text,
                    "evidence_text": evidence_text,
                    "label": label
                })

                for idx in sent_ids:
                    gold_sentence_pairs.add(
                        (doc_id, idx)
                    )
        # Negative / NOT_ENOUGH_INFO

        negative_candidates = []

        for doc_id_str, ev_list in evidence.items():

            doc_id = int(doc_id_str)

            paper = corpus_by_id.get(doc_id)

            if paper is None:
                continue

            for idx, sentence in enumerate(
                paper["abstract"]
            ):

                if (
                    doc_id,
                    idx
                ) not in gold_sentence_pairs:

                    negative_candidates.append(
                        sentence
                    )

        # take only a few negatives
        for sentence in negative_candidates[
            :max_negative_per_claim
        ]:

            examples.append({
                "claim": claim_text,
                "evidence_text": sentence,
                "label": "NOT_ENOUGH_INFO"
            })

    return examples

In [ ]:
train_examples = build_nli_examples(
    claims_trains,
    corpus_by_id
)

dev_examples = build_nli_examples(
    claims_dev,
    corpus_by_id
)

print(
    "Train examples:",
    len(train_examples)
)

print(
    "Dev examples:",
    len(dev_examples)
)

Train examples: 2454
Dev examples: 899


In [ ]:
print(
    Counter(
        ex["label"]
        for ex in train_examples
    )
)

print(
    Counter(
        ex["label"]
        for ex in dev_examples
    )
)

Counter({'NOT_ENOUGH_INFO': 1497, 'SUPPORT': 616, 'CONTRADICT': 341})
Counter({'NOT_ENOUGH_INFO': 561, 'SUPPORT': 216, 'CONTRADICT': 122})


In [ ]:
#label mapping
label2id = {
    "SUPPORT": 0,
    "CONTRADICT": 1,
    "NOT_ENOUGH_INFO": 2
}

id2label = {
    v: k
    for k, v in label2id.items()
}

for ex in train_examples:
    ex["label_id"] = label2id[
        ex["label"]
    ]

for ex in dev_examples:
    ex["label_id"] = label2id[
        ex["label"]
    ]

In [ ]:
def tokenize_examples(
    examples,
    tokenizer,
    max_length=256
):

    claims = [
        ex["claim"]
        for ex in examples
    ]

    evidences = [
        ex["evidence_text"]
        for ex in examples
    ]

    labels = [
        ex["label_id"]
        for ex in examples
    ]

    encodings = tokenizer(
        claims,
        evidences,
        truncation=True,
        padding=True,
        max_length=max_length
    )

    return encodings, labels

In [ ]:
from transformers import (
    AutoModelForSequenceClassification
)

model_name = (
    "allenai/scibert_scivocab_uncased"
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [ ]:
train_encodings, train_labels = (
    tokenize_examples(
        train_examples,
        tokenizer
    )
)

dev_encodings, dev_labels = (
    tokenize_examples(
        dev_examples,
        tokenizer
    )
)

train_dataset = Dataset.from_dict({

    "input_ids":
        train_encodings["input_ids"],

    "attention_mask":
        train_encodings["attention_mask"],

    "labels":
        train_labels
})

dev_dataset = Dataset.from_dict({

    "input_ids":
        dev_encodings["input_ids"],

    "attention_mask":
        dev_encodings["attention_mask"],

    "labels":
        dev_labels
})

In [ ]:
#جربت 3 fine-tuning strategies مختلفة وقارنت بين نتايجهم:
#Full freeze → F1 = 0.588
#LoRA → F1 = 0.619
#Partial fine-tuning → F1 = 0.696

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():

    if (
        "encoder.layer.10" in name
        or "encoder.layer.11" in name
        or "pooler" in name
        or "classifier" in name
    ):

        param.requires_grad = True

In [ ]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Trainable parameters: "
    f"{trainable:,} / {total:,}"
)

Trainable parameters: 14,768,643 / 109,920,771


In [ ]:
training_args = TrainingArguments(
    output_dir="./results_nli",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
    fp16=True,
    tf32=False,
    report_to="none"
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_macro": f1
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
train_result=trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,0.512763,0.616401,0.699666,0.535562,0.533012,0.532912
2,0.477619,0.600086,0.719689,0.589651,0.598347,0.592606
3,0.433371,0.562176,0.731924,0.573350,0.569703,0.569864
4,0.357358,0.580111,0.743048,0.604125,0.603920,0.604016
5,0.252960,0.610039,0.738598,0.605051,0.616979,0.609798
6,0.192067,0.594543,0.763070,0.624446,0.600609,0.609903
7,0.222507,0.630026,0.745273,0.617601,0.630994,0.623077
8,0.299533,0.604171,0.766407,0.629147,0.631342,0.629557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
dev_results = trainer.evaluate()
print(dev_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Macro
0.299533,0.604171,8,0.766407,0.629147,0.631342,0.629557


{'eval_loss': 0.6041710376739502, 'eval_accuracy': 0.7664071190211346, 'eval_precision': 0.6291473962043582, 'eval_recall': 0.6313422178497717, 'eval_f1_macro': 0.6295565029418547}


In [ ]:
model.save_pretrained(
    "./final_model"
)
tokenizer.save_pretrained(
    "./final_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

### Evaluate on Training Data

In [ ]:
train_results = trainer.evaluate(train_dataset)
print(train_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Macro
0.299533,0.187458,8,0.922575,0.877338,0.864882,0.869902


{'eval_loss': 0.18745771050453186, 'eval_accuracy': 0.9225753871230644, 'eval_precision': 0.8773379644574636, 'eval_recall': 0.8648816391802363, 'eval_f1_macro': 0.8699019407093941}


### Prepare Test Data and Evaluate

In [ ]:
claims_test = load_claims("data/claims_test.jsonl")

test_examples = build_nli_examples(
    claims_test,
    corpus_by_id
)

if not test_examples:
    print("Warning: claims_test.jsonl does not contain 'evidence' field for NLI. Generating placeholder NLI examples.")
    for claim_obj in claims_test:
        test_examples.append({
            "claim": claim_obj["claim"],
            "evidence_text": claim_obj["claim"], 
            "label": "NOT_ENOUGH_INFO", 
            "label_id": label2id["NOT_ENOUGH_INFO"] 
        })
for ex in test_examples:
    if "label_id" not in ex:
        ex["label_id"] = label2id[
            ex["label"]
        ]
test_encodings, test_labels = (
    tokenize_examples(
        test_examples,
        tokenizer
    )
)

test_dataset = Dataset.from_dict({

    "input_ids":
        test_encodings["input_ids"],

    "attention_mask":
        test_encodings["attention_mask"],

    "labels":
        test_labels
})

print("Test examples:", len(test_examples))
print("Test dataset size:", len(test_dataset))

Test examples: 300
Test dataset size: 300


In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Macro
0.299533,3.877581,8,0.093333,0.333333,0.031111,0.056911


{'eval_loss': 3.8775813579559326, 'eval_accuracy': 0.09333333333333334, 'eval_precision': 0.3333333333333333, 'eval_recall': 0.031111111111111114, 'eval_f1_macro': 0.05691056910569106}


In [ ]:
print(Counter([ex["label"] for ex in test_examples]))

Counter({'NOT_ENOUGH_INFO': 300})


In [ ]:
dev_results = trainer.evaluate(dev_dataset)
print(dev_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1 Macro
0.299533,0.604171,8,0.766407,0.629147,0.631342,0.629557


{'eval_loss': 0.6041710376739502, 'eval_accuracy': 0.7664071190211346, 'eval_precision': 0.6291473962043582, 'eval_recall': 0.6313422178497717, 'eval_f1_macro': 0.6295565029418547}


In [ ]:
def predict_verdict(claim_text, evidence_text, model=model, tokenizer=tokenizer, id2label=id2label):
    inputs = tokenizer(claim_text, evidence_text, truncation=True, padding=True, max_length=256, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

    return {
        "verdict": id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "all_probs": {id2label[i]: float(probs[i]) for i in range(len(probs))}
    }

In [ ]:
example = dev_examples[1]
result = predict_verdict(example["claim"], example["evidence_text"])
print("Claim:", example["claim"])
print("Evidence:", example["evidence_text"])
print("True label:", example["label"])
print("Predicted:", result)

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
Evidence: In conclusion, uncommon or rare genetic variants can easily create synthetic associations that are credited to common variants, and this possibility requires careful consideration in the interpretation and follow up of GWAS signals.
True label: SUPPORT
Predicted: {'verdict': 'SUPPORT', 'confidence': 0.6055186986923218, 'all_probs': {'SUPPORT': 0.6055186986923218, 'CONTRADICT': 0.38301634788513184, 'NOT_ENOUGH_INFO': 0.011464881710708141}}
